# De-Haze: train U-Net and Pix2Pix on Kaggle

Before running:
1. Settings -> Accelerator: **GPU T4 x2** (the code uses one GPU).
2. Add Input: the **code** dataset (zip from `python kaggle/make_code_zip.py`) and the **images** dataset
   (with `Hazy_Full/` and `GT_Full/`).
3. Do not `pip install -r requirements.txt` here: Kaggle already has TensorFlow, and that file would change its version.

In [ ]:
# 1. GPUs and versions
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
!nvidia-smi -L

In [ ]:
# 2. Paths. The attached datasets are found by searching /kaggle/input,
#    so the exact mount path does not matter.
import os
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/De-Haze")   # code runs here; /kaggle/working is saved with the notebook
CODE_DIR = None    # set by hand if the search finds more than one copy of the code
# To continue an interrupted run: attach the earlier notebook output and set this, e.g.
# RESUME_CHECKPOINTS = Path("/kaggle/input/<earlier-output>/De-Haze/outputs/checkpoints")
RESUME_CHECKPOINTS = None

def find_dirs(test):
    return [Path(root) for root, dirs, files in os.walk(INPUT, followlinks=True) if test(dirs, files)]

if CODE_DIR is None:
    found = find_dirs(lambda dirs, files: "train.py" in files and "config.py" in files)
    print("code folders found:", found)
    assert len(found) == 1, "Need exactly one code folder; set CODE_DIR by hand"
    CODE_DIR = found[0]
found = find_dirs(lambda dirs, files: "Hazy_Full" in dirs and "GT_Full" in dirs)
print("image folders found:", found)
assert found, "Attach the images dataset (it must contain Hazy_Full/ and GT_Full/)"
DATA_DIR = found[0]
print("CODE_DIR =", CODE_DIR)
print("DATA_DIR =", DATA_DIR)

In [ ]:
# 3. Copy the code (small) to the writable working folder
import shutil
shutil.copytree(CODE_DIR, WORK, dirs_exist_ok=True)
if RESUME_CHECKPOINTS:
    shutil.copytree(RESUME_CHECKPOINTS, WORK / "outputs" / "checkpoints", dirs_exist_ok=True)
    print("Copied checkpoints, training will resume from them")
os.chdir(WORK)
print(sorted(p.name for p in WORK.iterdir()))

In [ ]:
# 4. Link the images into dataset/ (a symlink, not a 1.5 GB copy) and check the split file
(WORK / "dataset").mkdir(exist_ok=True)
for name in ["Hazy_Full", "GT_Full", "Hazy_Small", "GT_Small"]:
    src, dst = DATA_DIR / name, WORK / "dataset" / name
    if src.exists() and not dst.exists():
        os.symlink(src, dst, target_is_directory=True)
for name in ["Hazy_Full", "GT_Full"]:
    print(name, len(os.listdir(WORK / "dataset" / name)), "files (expected 2036)")
assert (WORK / "splits" / "split_v1.json").exists(), "splits/split_v1.json missing from the code dataset"

In [ ]:
# 5. Train U-Net (checkpoint every epoch; running this cell again resumes)
!python train.py --model unet --epochs 30 --max-hours 5

In [ ]:
# 6. Train Pix2Pix
!python train.py --model pix2pix --epochs 20 --max-hours 5

In [ ]:
# 7. Zip the models and logs to /kaggle/working/weights_all.zip
import zipfile
models_dir = WORK / "outputs" / "models"
out = Path("/kaggle/working/weights_all.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(models_dir.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(models_dir.parent))
            print(" ", f.relative_to(models_dir.parent), f"{f.stat().st_size / 1e6:.0f} MB")
print(out, f"{out.stat().st_size / 1e6:.0f} MB")